In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import (
    IntSlider,
    HTML,
    HTMLMath,
    VBox,
    HBox,
    Layout
)

from IPython.display import display

# ============================================================
# TWO-DIMENSIONAL FOURIER SERIES
#
# f(x,y) =
# sum_m sum_n A_mn exp[i 2pi(mx/T1 + ny/T2)]
#
# The Fourier coefficients are calculated numerically
# from one complete period using the 2D FFT.
# ============================================================

plt.ioff()

# ============================================================
# JUPYTER / BINDER DISPLAY SETTINGS
# ============================================================

display(HTML("""
<style>

.container {
    width:98% !important;
    max-width:none !important;
}

.output_area,
.output_subarea {
    max-width:none !important;
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
}

.output_scroll {
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
    box-shadow:none !important;
}

.jp-Cell-outputWrapper,
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    max-width:none !important;
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
}

.widget-output,
.jupyter-widgets-output-area,
.widget-box {
    max-width:none !important;
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
}

.jupyter-matplotlib,
.jupyter-matplotlib-figure {
    overflow:visible !important;
    resize:none !important;
}

.jupyter-matplotlib::-webkit-resizer,
.jupyter-matplotlib-figure::-webkit-resizer {
    display:none !important;
}

.fs2-title {
    font-family:Arial, sans-serif;
    font-size:20px;
    font-weight:bold;
    color:#6f3fa0;
}

.fs2-label {
    font-family:Arial, sans-serif;
    font-size:14px;
    font-weight:bold;
}

.fs2-value {
    font-family:Arial, sans-serif;
    font-size:14px;
    font-weight:bold;
    color:#0b3d91;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    width:1200px;
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.48;
    margin-bottom:10px;
">

<div class="fs2-title" style="margin-bottom:8px;">
Two-Dimensional Fourier Series
</div>

<div style="margin-bottom:5px;">
A periodic function of two variables can be represented by a
double exponential Fourier series with coefficients A<sub>mn</sub>.
The two indices m and n correspond to harmonics in the x and y
directions, respectively.
</div>

<div style="margin-bottom:5px;">
This notebook uses one period of a two-dimensional rectangular pulse.
Its Fourier coefficients are calculated numerically from the sampled
function. The coefficients are not entered manually.
</div>

<div style="margin-bottom:5px;">
The sliders M and N determine how many positive and negative Fourier
indices are retained along the x and y directions. Increasing M improves
the representation of variation along x, while increasing N improves
the representation along y.
</div>

<div>
<b>This notebook:</b> compares the original periodic function with its
truncated two-dimensional Fourier-series reconstruction and displays
the magnitude of the retained Fourier coefficients.
</div>

</div>
""")

# ============================================================
# MATHEMATICAL RELATIONS
# ============================================================

series_math = HTMLMath(
    value=(
        r'\('
        r'f(x,y)'
        r'\approx'
        r'\displaystyle'
        r'\sum_{m=-M}^{M}'
        r'\sum_{n=-N}^{N}'
        r'A_{mn}'
        r'\exp\left['
        r'i2\pi'
        r'\left('
        r'\frac{mx}{T_1}'
        r'+'
        r'\frac{ny}{T_2}'
        r'\right)'
        r'\right]'
        r'\)'
    )
)

coeff_math = HTMLMath(
    value=(
        r'\('
        r'A_{mn}'
        r'='
        r'\frac{1}{T_1T_2}'
        r'\displaystyle'
        r'\int_0^{T_1}'
        r'\int_0^{T_2}'
        r'f(x,y)'
        r'\exp\left['
        r'-i2\pi'
        r'\left('
        r'\frac{mx}{T_1}'
        r'+'
        r'\frac{ny}{T_2}'
        r'\right)'
        r'\right]'
        r'dx\,dy'
        r'\)'
    )
)

relations_row = VBox(
    [
        series_math,
        coeff_math
    ],
    layout=Layout(
        width='1100px',
        gap='4px',
        overflow='visible'
    )
)

# ============================================================
# FIXED PARAMETERS
# ============================================================

GRID_SIZE = 256

T1 = 2.0
T2 = 2.0

HALF_WIDTH_X = 0.45
HALF_WIDTH_Y = 0.30

# ============================================================
# SPATIAL AXES
#
# One complete period:
#
# -T1/2 <= x < T1/2
# -T2/2 <= y < T2/2
# ============================================================

x_axis = np.linspace(
    -T1 / 2.0,
    T1 / 2.0,
    GRID_SIZE,
    endpoint=False
)

y_axis = np.linspace(
    -T2 / 2.0,
    T2 / 2.0,
    GRID_SIZE,
    endpoint=False
)

X, Y = np.meshgrid(
    x_axis,
    y_axis
)

# ============================================================
# ORIGINAL PERIODIC FUNCTION
#
# One period contains a rectangular pulse.
# Repetition of this cell generates a 2D periodic function.
# ============================================================

original_function = (
    (np.abs(X) <= HALF_WIDTH_X)
    &
    (np.abs(Y) <= HALF_WIDTH_Y)
).astype(float)

# ============================================================
# 2D FOURIER-SERIES COEFFICIENTS
#
# For a uniformly sampled period, the normalized 2D DFT gives
# the discrete approximation of the Fourier-series coefficients.
# ============================================================

coefficients = np.fft.fftshift(
    np.fft.fft2(
        original_function
    )
) / (GRID_SIZE**2)

# ============================================================
# INTEGER FOURIER INDICES
# ============================================================

fourier_indices = np.arange(
    -GRID_SIZE // 2,
    GRID_SIZE // 2
)

# ============================================================
# SLIDER STYLE
# ============================================================

slider_style = {
    'description_width': '0px'
}

slider_layout = Layout(
    width='240px'
)

label_layout = Layout(
    width='120px',
    min_width='120px'
)

value_layout = Layout(
    width='60px',
    min_width='60px',
    margin='0px 0px 0px 6px'
)

row_layout = Layout(
    width='440px',
    height='40px',
    align_items='center'
)

# ============================================================
# M SLIDER
# ============================================================

M_slider = IntSlider(
    min=0,
    max=20,
    step=1,
    value=5,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

M_label = HTML(
    '<div class="fs2-label">Harmonics M:</div>',
    layout=label_layout
)

M_value = HTML(
    '<div class="fs2-value">5</div>',
    layout=value_layout
)

M_row = HBox(
    [
        M_label,
        M_slider,
        M_value
    ],
    layout=row_layout
)

# ============================================================
# N SLIDER
# ============================================================

N_slider = IntSlider(
    min=0,
    max=20,
    step=1,
    value=5,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

N_label = HTML(
    '<div class="fs2-label">Harmonics N:</div>',
    layout=label_layout
)

N_value = HTML(
    '<div class="fs2-value">5</div>',
    layout=value_layout
)

N_row = HBox(
    [
        N_label,
        N_slider,
        N_value
    ],
    layout=row_layout
)

# ============================================================
# CONTROLS PANEL
# ============================================================

controls_panel = VBox(
    [
        HTML("""
        <div class="fs2-title" style="margin-bottom:8px;">
            Fourier-Series Parameters
        </div>
        """),

        M_row,
        N_row
    ],
    layout=Layout(
        width='470px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# NUMERICAL INFORMATION
# ============================================================

terms_math = HTMLMath()
rmse_math = HTMLMath()
max_error_math = HTMLMath()

information_panel = VBox(
    [
        HTML("""
        <div class="fs2-title" style="margin-bottom:8px;">
            Reconstruction Information
        </div>
        """),

        terms_math,
        rmse_math,
        max_error_math
    ],
    layout=Layout(
        width='650px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# TOP ROW
# ============================================================

top_row = HBox(
    [
        controls_panel,
        information_panel
    ],
    layout=Layout(
        width='1140px',
        gap='15px',
        align_items='stretch',
        overflow='visible'
    )
)

# ============================================================
# RECONSTRUCTION FUNCTION
# ============================================================

def reconstruct_fourier_series(M, N):

    # --------------------------------------------------------
    # Mask in shifted coefficient plane
    #
    # rows    -> n
    # columns -> m
    # --------------------------------------------------------

    mask = np.zeros(
        (
            GRID_SIZE,
            GRID_SIZE
        ),
        dtype=float
    )

    center = (
        GRID_SIZE // 2
    )

    row_start = (
        center - N
    )

    row_end = (
        center + N + 1
    )

    column_start = (
        center - M
    )

    column_end = (
        center + M + 1
    )

    mask[
        row_start:row_end,
        column_start:column_end
    ] = 1.0

    # --------------------------------------------------------
    # Retained Fourier coefficients
    # --------------------------------------------------------

    retained_coefficients = (
        coefficients
        *
        mask
    )

    # --------------------------------------------------------
    # Reconstruction
    #
    # coefficients were normalized by GRID_SIZE^2, so we
    # restore the FFT scaling before inverse transformation.
    # --------------------------------------------------------

    reconstruction = np.fft.ifft2(
        np.fft.ifftshift(
            retained_coefficients
            *
            GRID_SIZE**2
        )
    )

    reconstruction = np.real(
        reconstruction
    )

    return (
        reconstruction,
        retained_coefficients
    )

# ============================================================
# INITIAL RECONSTRUCTION
# ============================================================

(
    reconstruction_initial,
    retained_initial
) = reconstruct_fourier_series(
    M_slider.value,
    N_slider.value
)

# ============================================================
# FIGURE 1
# ORIGINAL FUNCTION
# ============================================================

fig_original, ax_original = plt.subplots(
    figsize=(3.8, 3.8)
)

fig_original.canvas.header_visible = False
fig_original.canvas.footer_visible = False
fig_original.canvas.toolbar_visible = False

fig_original.canvas.layout = Layout(
    width='380px',
    height='380px',
    overflow='visible'
)

ax_original.set_title(
    'Original Periodic Function',
    fontsize=13,
    fontweight='bold',
    color='#6f3fa0'
)

ax_original.set_xlabel(
    'x',
    fontsize=10
)

ax_original.set_ylabel(
    'y',
    fontsize=10
)

original_image = ax_original.imshow(
    original_function,
    extent=[
        -T1 / 2.0,
        T1 / 2.0,
        -T2 / 2.0,
        T2 / 2.0
    ],
    origin='lower',
    aspect='equal',
    vmin=0.0,
    vmax=1.0,
    interpolation='nearest'
)

fig_original.subplots_adjust(
    left=0.15,
    right=0.97,
    top=0.88,
    bottom=0.13
)

# ============================================================
# FIGURE 2
# FOURIER-SERIES RECONSTRUCTION
# ============================================================

fig_reconstruction, ax_reconstruction = plt.subplots(
    figsize=(3.8, 3.8)
)

fig_reconstruction.canvas.header_visible = False
fig_reconstruction.canvas.footer_visible = False
fig_reconstruction.canvas.toolbar_visible = False

fig_reconstruction.canvas.layout = Layout(
    width='380px',
    height='380px',
    overflow='visible'
)

ax_reconstruction.set_title(
    'Reconstruction',
    fontsize=13,
    fontweight='bold',
    color='#0b3d91'
)

ax_reconstruction.set_xlabel(
    'x',
    fontsize=10
)

ax_reconstruction.set_ylabel(
    'y',
    fontsize=10
)

reconstruction_image = ax_reconstruction.imshow(
    reconstruction_initial,
    extent=[
        -T1 / 2.0,
        T1 / 2.0,
        -T2 / 2.0,
        T2 / 2.0
    ],
    origin='lower',
    aspect='equal',
    vmin=-0.25,
    vmax=1.25,
    interpolation='bilinear'
)

fig_reconstruction.subplots_adjust(
    left=0.15,
    right=0.97,
    top=0.88,
    bottom=0.13
)

# ============================================================
# FIGURE 3
# FOURIER COEFFICIENT MAGNITUDES
# ============================================================

fig_coeff, ax_coeff = plt.subplots(
    figsize=(3.8, 3.8)
)

fig_coeff.canvas.header_visible = False
fig_coeff.canvas.footer_visible = False
fig_coeff.canvas.toolbar_visible = False

fig_coeff.canvas.layout = Layout(
    width='380px',
    height='380px',
    overflow='visible'
)

ax_coeff.set_title(
    'Retained Fourier Coefficients |Aₘₙ|',
    fontsize=13,
    fontweight='bold',
    color='#16802b'
)

ax_coeff.set_xlabel(
    'm',
    fontsize=10
)

ax_coeff.set_ylabel(
    'n',
    fontsize=10
)

# ------------------------------------------------------------
# Display only a limited central coefficient region
# ------------------------------------------------------------

COEFF_LIMIT = 22

center = (
    GRID_SIZE // 2
)

coeff_display_initial = np.abs(
    retained_initial[
        center - COEFF_LIMIT:center + COEFF_LIMIT + 1,
        center - COEFF_LIMIT:center + COEFF_LIMIT + 1
    ]
)

coeff_image = ax_coeff.imshow(
    coeff_display_initial,
    extent=[
        -COEFF_LIMIT,
        COEFF_LIMIT,
        -COEFF_LIMIT,
        COEFF_LIMIT
    ],
    origin='lower',
    aspect='equal',
    vmin=0.0,
    vmax=np.max(
        np.abs(
            coefficients
        )
    ),
    interpolation='nearest'
)

ax_coeff.set_xlim(
    -COEFF_LIMIT,
    COEFF_LIMIT
)

ax_coeff.set_ylim(
    -COEFF_LIMIT,
    COEFF_LIMIT
)

fig_coeff.subplots_adjust(
    left=0.15,
    right=0.97,
    top=0.88,
    bottom=0.13
)

# ============================================================
# FIGURES ROW
# ============================================================

figures_row = HBox(
    [
        fig_original.canvas,
        fig_reconstruction.canvas,
        fig_coeff.canvas
    ],
    layout=Layout(
        width='1160px',
        gap='8px',
        align_items='flex-start',
        overflow='visible'
    )
)

# ============================================================
# INTERPRETATION PANEL
# ============================================================

interpretation_panel = HTML("""
<div style="
    width:1140px;
    padding:9px 12px;
    border:1px solid #d7c7e5;
    font-family:Arial, sans-serif;
    font-size:14px;
    line-height:1.52;
    box-sizing:border-box;
">

<b style="color:#6f3fa0;">
Key observations:
</b>

Increasing M retains more Fourier components in the horizontal
frequency direction, while increasing N retains more components
in the vertical frequency direction. The reconstructed function
therefore approaches the original two-dimensional periodic function
as both values increase. Near the discontinuities, oscillatory
overshoots remain visible because of the two-dimensional analogue
of the Gibbs phenomenon.

</div>
""")

# ============================================================
# UPDATE FUNCTION
#
# No clear_output()
# No figure recreation
# No axis rescaling
#
# Only existing images and displayed numerical values
# are updated.
# ============================================================

def update_notebook(change=None):

    M = (
        M_slider.value
    )

    N = (
        N_slider.value
    )

    # --------------------------------------------------------
    # Slider values
    # --------------------------------------------------------

    M_value.value = (
        f'<div class="fs2-value">{M}</div>'
    )

    N_value.value = (
        f'<div class="fs2-value">{N}</div>'
    )

    # --------------------------------------------------------
    # Reconstruction
    # --------------------------------------------------------

    (
        reconstruction,
        retained_coefficients
    ) = reconstruct_fourier_series(
        M,
        N
    )

    # --------------------------------------------------------
    # Numerical error
    # --------------------------------------------------------

    error = (
        original_function
        -
        reconstruction
    )

    rmse = np.sqrt(
        np.mean(
            error**2
        )
    )

    maximum_error = np.max(
        np.abs(
            error
        )
    )

    number_of_terms = (
        (2 * M + 1)
        *
        (2 * N + 1)
    )

    # --------------------------------------------------------
    # Update reconstruction image
    # --------------------------------------------------------

    reconstruction_image.set_data(
        reconstruction
    )

    # --------------------------------------------------------
    # Update retained coefficient image
    # --------------------------------------------------------

    coeff_display = np.abs(
        retained_coefficients[
            center - COEFF_LIMIT:center + COEFF_LIMIT + 1,
            center - COEFF_LIMIT:center + COEFF_LIMIT + 1
        ]
    )

    coeff_image.set_data(
        coeff_display
    )

    # --------------------------------------------------------
    # Numerical information
    # --------------------------------------------------------

    terms_math.value = (
        r'\('
        r'(2M+1)(2N+1)='
        +
        str(number_of_terms)
        +
        r'\ \mathrm{retained\ coefficients}'
        r'\)'
    )

    rmse_math.value = (
        r'\('
        r'\mathrm{RMSE}='
        +
        f'{rmse:.6e}'
        +
        r'\)'
    )

    max_error_math.value = (
        r'\('
        r'\max|f-f_{MN}|='
        +
        f'{maximum_error:.6e}'
        +
        r'\)'
    )

    # --------------------------------------------------------
    # Redraw only
    # --------------------------------------------------------

    fig_original.canvas.draw_idle()
    fig_reconstruction.canvas.draw_idle()
    fig_coeff.canvas.draw_idle()

# ============================================================
# CONNECT CONTROLS
# ============================================================

M_slider.observe(
    update_notebook,
    names='value'
)

N_slider.observe(
    update_notebook,
    names='value'
)

# ============================================================
# INITIAL UPDATE
# ============================================================

update_notebook()

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        relations_row,
        top_row,
        figures_row,
        interpretation_panel
    ],
    layout=Layout(
        width='1180px',
        gap='10px',
        overflow='visible'
    )
)

display(
    main_layout
)